
**This notebook runs HBV over CAMELS-FR dataset catchments with less than 10% of missing data, and stock results (parameters and performance metrics) in HBV_Simulation_Data_CAMELS_FR.csv**

**Author:** Lionel Cedric Gohouede

## 1. MOUNT GOOGLE DRIVE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. IMPORT LIBRARIES

In [ ]:
pip install aqua-fetch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.1/289.1 kB 5.5 MB/s eta 0:00:00


In [ ]:
!pip install git+https://github.com/kratzert/RRMPG.git

  Cloning https://github.com/kratzert/RRMPG.git to /tmp/pip-req-build-4ump7_7_
  Running command git clone --filter=blob:none --quiet https://github.com/kratzert/RRMPG.git /tmp/pip-req-build-4ump7_7_
  Resolved https://github.com/kratzert/RRMPG.git to commit 7de78c25acc1c255d2acaf739d65e9ce7bbd60c3
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 913.3/913.3 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 2.1 MB/s eta 0:00:00
  Created wheel for rrmpg: filename=rrmpg-0.1.1-py3-none-any.whl size=609986 sha256=1c286a7a474f2ef4baeb7eb2cc0efe2dbe6532fccfd12cb375fa998e417019cc
  Stored in directory: /tmp/pip-ephem-wheel-cache-nbyq3a2f/wheels/7d/ec/25/408ea8f9d8a1aff931ffd2c082074611b43cc13f5e

In [ ]:
# === 1. Standard Library Imports ===
import math
import warnings

# === 2. Third-Party Library Imports ===
from google.colab import drive
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
from numba import njit
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import xarray as xr
from aqua_fetch import RainfallRunoff

# === 3. Configurations ===
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully and cleaned up!")

✅ Libraries imported successfully and cleaned up!


## 3. FILTER DATA BY TIME PERIOD (1990-2014)

In [ ]:
from aqua_fetch import RainfallRunoff
import numpy as np
import pandas as pd

# ============================================
# Load CAMELS-FR
# ============================================
rr = RainfallRunoff("CAMELS_FR")

_, ds = rr.fetch()
meta = rr.fetch_static_features()

# ============================================
# Time period
# ============================================
start_date = "2000-01-01"
end_date   = "2021-12-31"

print("Loading, filtering, and aligning CAMELS-FR data...\n")

ds = ds.sel(time=slice(start_date, end_date)).load()

# ============================================
# Station IDs
# ============================================
stations = list(ds.data_vars)

# ============================================
# Dynamic feature indices
# ============================================
feature_index = {
    name: i
    for i, name in enumerate(ds.dynamic_features.values)
}

pcp_idx = feature_index["pcp_mm"]
pet_idx = feature_index["pet_mm_pm"]   # Penman-Monteith PET
q_idx   = feature_index["q_cms_obs"]   # Discharge stored in l/s → converted below

time_index = pd.to_datetime(ds.time.values)

# ============================================
# Fast extraction to DataFrames
# ============================================
df_pcp = pd.DataFrame(
    {st: ds[st].values[:, pcp_idx] for st in stations},
    index=time_index,
)

df_pet = pd.DataFrame(
    {st: ds[st].values[:, pet_idx] for st in stations},
    index=time_index,
)

# CAMELS-France stores discharge in l/s → divide by 1 000 to get m³/s
df_q = pd.DataFrame(
    {st: ds[st].values[:, q_idx] for st in stations},
    index=time_index,
) / 1_000.0

# ============================================
# Static attributes (UPDATED WITH FALLBACKS)
# ============================================
meta.index = meta.index.astype(str).str.strip()

# Cascade through available area columns to fill NaNs
df_area = (
    meta["area_km2"]
    .fillna(meta["sta_area_snap"])
    .fillna(meta["sit_area_hydro"])
).copy()

# Rename the series back to 'area_km2' to avoid breaking downstream logic
df_area.name = "area_km2"
df_area.index = df_area.index.astype(str).str.strip()

# ============================================
# Ensure station IDs are consistent
# ============================================
df_pcp.columns = df_pcp.columns.astype(str).str.strip()
df_pet.columns = df_pet.columns.astype(str).str.strip()
df_q.columns   = df_q.columns.astype(str).str.strip()

# ============================================
# Remove stations without catchment area
# ============================================
n_missing_area = df_area.isna().sum()
print(f"⚠️  Stations dropped (missing ALL area data) : {n_missing_area}")
df_area = df_area.dropna()

# ============================================
# Keep only common stations
# ============================================
common_stations = sorted(
    set(df_pcp.columns)
    & set(df_pet.columns)
    & set(df_q.columns)
    & set(df_area.index)
)

df_pcp  = df_pcp[common_stations]
df_pet  = df_pet[common_stations]
df_q    = df_q[common_stations]
df_area = df_area.loc[common_stations]

print(f"✅ Dynamic stations : {len(df_pcp.columns)}")
print(f"✅ Static stations  : {len(df_area)}")
print(f"✅ Common stations  : {len(common_stations)}")

# ============================================
# Memory-Optimised Wrapper Classes
# ============================================
class SimpleArray:
    __slots__ = ["array"]

    def __init__(self, array):
        self.array = array

    def to_numpy(self):
        return self.array


class StationData:
    __slots__ = ["data"]

    def __init__(self, data):
        self.data = data

    def sel(self, dynamic_features=None):
        return SimpleArray(self.data[dynamic_features])

    def static(self, feature):
        return self.data[feature]


# ============================================
# Convert to NumPy once
# ============================================
pcp_arr  = df_pcp.to_numpy(dtype=np.float64)
pet_arr  = df_pet.to_numpy(dtype=np.float64)
q_arr    = df_q.to_numpy(dtype=np.float64)   # m³/s
area_arr = df_area.to_numpy(dtype=np.float64)
date_arr = time_index.to_numpy()

# ============================================
# Build ds_recent
# ============================================
print("\nBuilding ds_recent dictionary...")

ds_recent = {
    st: StationData({
        "pcp_mm":    pcp_arr[:, i],
        "pet_mm":    pet_arr[:, i],
        "q_cms_obs": q_arr[:, i],    # m³/s
        "area_km2":  area_arr[i],
        "date":      date_arr,
    })
    for i, st in enumerate(common_stations)
}

print("✅ Dictionary built successfully!")

# ============================================
# Quick sanity check
# ============================================
all_stations = common_stations
b1_ratio = 0.7
max_missing_ratio = 0.1
results = {}

test_station = all_stations[0]

Q_obs_cms = ds_recent[test_station].sel("q_cms_obs").to_numpy()
P         = ds_recent[test_station].sel("pcp_mm").to_numpy()
PET       = ds_recent[test_station].sel("pet_mm").to_numpy()
area      = ds_recent[test_station].static("area_km2")

print(f"\nTesting station : {test_station}")
print(f"Q shape         : {Q_obs_cms.shape}  (m³/s)")
print(f"P shape         : {P.shape}")
print(f"PET shape       : {PET.shape}")
print(f"Area            : {area:.2f} km²")
print(f"Missing Q       : {np.isnan(Q_obs_cms).sum()} ({100 * np.isnan(Q_obs_cms).mean():.2f}%)")

downloading 6 files to /usr/local/lib/python3.12/dist-packages/aqua_fetch/data/CAMELS/CAMELS_FR
downloading ADDITIONAL_LICENSES.zip
0% of 0.07 MB downloaded
100% of 0.07 MB downloaded
downloading CAMELS_FR_attributes.zip
0% of 9.88 MB downloaded
100% of 9.88 MB downloaded
downloading CAMELS_FR_geography.zip
0% of 1.45 MB downloaded
100% of 1.45 MB downloaded
downloading CAMELS_FR_time_series.zip
0% of 361.39 MB downloaded
20% of 361.39 MB downloaded
40% of 361.39 MB downloaded
60% of 361.39 MB downloaded
80% of 361.39 MB downloaded
100% of 361.39 MB downloaded
downloading README.md
0% of 0.01 MB downloaded
100% of 0.01 MB downloaded
downloading CAMELS-FR_description.ods
0% of 0.05 MB downloaded
100% of 0.05 MB downloaded
unzipping files in /usr/local/lib/python3.12/dist-packages/aqua_fetch/data/CAMELS/CAMELS_FR
unzipping ADDITIONAL_LICENSES.zip to ADDITIONAL_LICENSES
unzipping CAMELS_FR_time_series.zip to CAMELS_FR_time_series
unzipping CAMELS_FR_geography.zip to CAMELS_FR_geography
un

## 4. MAIN CODE

In [ ]:
import numpy as np
import pandas as pd
from numba import njit
from scipy.optimize import minimize

# Unit conversion factor
CONV = 86.4   # (1 mm/day * 1 km² = 1/86.4 m³/s)

# ============================================
# 1. COMPILED HBV MODEL
# ============================================
@njit(cache=True)
def hbv_numba(P, ET, params, Qsim):
    """
    Runs the HBV bucket model in place into the preallocated `Qsim` buffer.
    """
    FC, BETA, LP, K0, K1, K2, UZL, PERC = (
        params[0], params[1], params[2], params[3],
        params[4], params[5], params[6], params[7]
    )
    SM = 0.5 * FC
    UZ = 0.0
    LZ = 0.0
    n = len(P)

    for t in range(n):
        p = P[t]
        et = ET[t]

        if np.isnan(p) or np.isnan(et):
            Qsim[t] = np.nan
            continue

        soil_ratio = max(0.0, min(SM / FC, 1.0))
        recharge = p * soil_ratio ** BETA
        SM += p - recharge
        if SM > FC:
            recharge += SM - FC
            SM = FC
        SM = max(SM, 0.0)

        evap = min(et * min(SM / (LP * FC), 1.0), SM)
        SM -= evap
        SM = max(SM, 0.0)

        UZ += recharge
        perc = min(PERC, UZ)
        UZ -= perc
        LZ += perc

        Q0 = K0 * (UZ - UZL) if UZ > UZL else 0.0
        UZ -= Q0

        Q1 = K1 * UZ
        UZ -= Q1

        Q2 = K2 * LZ
        LZ -= Q2

        UZ = max(UZ, 0.0)
        LZ = max(LZ, 0.0)

        Qsim[t] = max(0.0, Q0 + Q1 + Q2)

    return Qsim  # mm/day


# ============================================
# 2. HBV OBJECTIVE
# ============================================
def objective_hbv(x, P_train, ET_train,
                  obs_train_cms_filled, mask_train_f,
                  denom_train_cms, area):
    n = len(P_train)
    Qsim_mmd = np.empty(n, dtype=np.float64)

    try:
        hbv_numba(P_train, ET_train, np.asarray(x, dtype=np.float64), Qsim_mmd)
    except Exception:
        return 1e9

    Qsim_mmd = np.where(np.isnan(Qsim_mmd), 0.0, Qsim_mmd)
    Qsim_cms = Qsim_mmd * area / CONV

    if denom_train_cms <= 0.0 or Qsim_cms.shape[0] != obs_train_cms_filled.shape[0]:
        return 1e9

    diff = (Qsim_cms - obs_train_cms_filled) * mask_train_f
    nse  = 1.0 - np.sum(diff ** 2) / denom_train_cms

    return 1e9 if not np.isfinite(nse) else 1.0 - nse


def runoff_ratio(Q_mmd, P_mmd):
    mask  = ~np.isnan(Q_mmd) & ~np.isnan(P_mmd)
    sum_Q = np.sum(Q_mmd[mask])
    sum_P = np.sum(P_mmd[mask])
    return np.nan if sum_P == 0.0 else sum_Q / sum_P


# ============================================
# 3. METRIC SUITE
# ============================================
def compute_all_metrics(obs_cms, sim_cms):
    mask = ~np.isnan(obs_cms) & ~np.isnan(sim_cms)
    o = obs_cms[mask]
    s = sim_cms[mask]
    L = o.size

    if L == 0:
        return dict(NSE=np.nan, NNSE=np.nan, RMSE=np.nan,
                    PBIAS=np.nan, FHV=np.nan, FLV=np.nan, KGE=np.nan)

    obs_mean = np.mean(o)
    sim_mean = np.mean(s)
    denom    = np.sum((o - obs_mean) ** 2)

    if denom == 0.0:
        nse = nnse = np.nan
    else:
        nse  = 1.0 - np.sum((s - o) ** 2) / denom
        nnse = (1.0 / (2.0 - nse)) if np.isfinite(nse) else np.nan

    rmse  = np.sqrt(np.mean((s - o) ** 2))
    sum_obs = np.sum(o)

    # PBIAS (%) per Moriasi et al. (2007) / Yilmaz Eq A1
    pbias = 100.0 * np.sum(s - o) / sum_obs if sum_obs != 0.0 else np.nan

    # Yilmaz et al. (2008) FDC tail metrics
    o_sorted = np.sort(o) # Ascending
    s_sorted = np.sort(s) # Ascending

    # Top-2 % high flows (FHV) - Yilmaz Eq A3
    n_hv       = max(1, int(round(0.02 * L)))
    sum_hv_obs = np.sum(o_sorted[-n_hv:])
    sum_hv_sim = np.sum(s_sorted[-n_hv:])
    fhv        = 100.0 * (sum_hv_sim - sum_hv_obs) / sum_hv_obs if sum_hv_obs != 0.0 else np.nan

    # Bottom-30 % low flows (FLV) - Yilmaz Eq A4 Strict Compliance
    n_lv = max(1, int(round(0.30 * L)))
    o_lv = o_sorted[:n_lv]
    s_lv = s_sorted[:n_lv]

    # Constant positive floor to stabilize logs
    eps  = 1e-6
    log_o_lv = np.log10(np.maximum(o_lv, eps))
    log_s_lv = np.log10(np.maximum(s_lv, eps))

    # Yilmaz Eq A4 requires subtracting the log of the minimum flow (index L)
    # Since arrays are sorted ascending, index 0 is the absolute minimum
    log_o_min = log_o_lv[0]
    log_s_min = log_s_lv[0]

    term_s = np.sum(log_s_lv - log_s_min)
    term_o = np.sum(log_o_lv - log_o_min)

    if term_o == 0.0:
        flv = np.nan
    else:
        # -1 * ( (Simulated Shifted Sum - Observed Shifted Sum) / Observed Shifted Sum ) * 100
        flv = -100.0 * (term_s - term_o) / term_o

    sigma_obs = np.std(o)
    sigma_sim = np.std(s)

    # KGE numerical robustness
    tol = 1e-12
    if sigma_obs < tol or sigma_sim < tol or abs(obs_mean) < tol:
        kge = np.nan
    else:
        r     = np.corrcoef(o, s)[0, 1]
        alpha = sigma_sim / sigma_obs
        beta  = sim_mean / obs_mean
        kge   = (1.0 - np.sqrt((r-1)**2 + (alpha-1)**2 + (beta-1)**2)
                 if np.isfinite(r) else np.nan)

    return dict(NSE=nse, NNSE=nnse, RMSE=rmse,
                PBIAS=pbias, FHV=fhv, FLV=flv, KGE=kge)

def compute_pte(obs_cms, sim_cms, dates):
    """
    Computes Mean Absolute Annual Peak Timing Error (PTE) in days.
    Groups by calendar year using an 80% minimum data completeness criterion.
    """
    df = pd.DataFrame({'obs': obs_cms, 'sim': sim_cms}, index=pd.DatetimeIndex(dates))
    df = df.dropna()

    if df.empty:
        return np.nan

    pte_list = []

    for year, group in df.groupby(df.index.year):
        # Minimum data completeness criterion based on 80% of days in that year
        days_in_year = pd.Timestamp(year, 12, 31).dayofyear
        if len(group) < 0.8 * days_in_year:
            continue

        idx_obs_peak = group['obs'].idxmax()
        idx_sim_peak = group['sim'].idxmax()

        diff_days = abs((idx_sim_peak - idx_obs_peak).days)
        pte_list.append(diff_days)

    return np.mean(pte_list) if pte_list else np.nan


# ============================================
# 4. EXECUTION LOOP
# ============================================
all_stations      = list(ds_recent.keys())
b1_ratio          = 0.7
max_missing_ratio = 0.1
results           = {}

param_names  = ["FC", "BETA", "LP", "K0", "K1", "K2", "UZL", "PERC"]
param_bounds = [
    (50.0, 700.0),   # FC
    (1.0, 6.0),      # BETA
    (0.3, 1.0),      # LP
    (0.05, 0.5),     # K0
    (0.01, 0.3),     # K1
    (0.001, 0.15),   # K2
    (0.0, 100.0),    # UZL
    (0.0, 6.0),      # PERC
]

bounds_low  = np.array([b[0] for b in param_bounds])
bounds_high = np.array([b[1] for b in param_bounds])

n_skip_no_area    = 0
n_skip_missing    = 0
n_skip_no_data    = 0
n_skip_calib_fail = 0

# Set random seed globally once for reproducible parameter exploration across all stations
np.random.seed(42)

for i, station_id in enumerate(all_stations, 1):
    print(f"\n=== Station {station_id}  ({i}/{len(all_stations)}) ===")

    # ── Load data ─────────────────────────────────────────────────────────────
    Q_obs_cms = ds_recent[station_id].sel(dynamic_features="q_cms_obs").to_numpy()
    P         = ds_recent[station_id].sel(dynamic_features="pcp_mm").to_numpy()
    ET        = ds_recent[station_id].sel(dynamic_features="pet_mm").to_numpy()
    area      = ds_recent[station_id].static("area_km2")
    times     = ds_recent[station_id].static("date")

    if area is None or not np.isfinite(area) or area <= 0.0:
        print("⚠️  Skipped — missing or invalid catchment area.")
        n_skip_no_area += 1
        continue

    N = len(Q_obs_cms)

    if N == 0 or np.all(np.isnan(Q_obs_cms)):
        print("⚠️  Skipped — no valid discharge data.")
        n_skip_no_data += 1
        continue

    missing_count = int(np.isnan(Q_obs_cms).sum())
    missing_ratio = missing_count / N
    if missing_ratio > max_missing_ratio:
        print(f"⚠️  Skipped — {missing_ratio*100:.1f}% missing values.")
        n_skip_missing += 1
        continue

    Q_obs_mmd = Q_obs_cms * CONV / area
    RR        = runoff_ratio(Q_obs_mmd, P)

    # ── Train / validation split ──────────────────────────────────────────────
    b1 = int(N * b1_ratio)

    Q_obs_train_cms = Q_obs_cms[:b1]
    P_train         = np.asarray(P[:b1], dtype=np.float64)
    ET_train        = np.asarray(ET[:b1], dtype=np.float64)

    mask_train = ~np.isnan(Q_obs_train_cms) & ~np.isnan(P_train) & ~np.isnan(ET_train)
    mask_train_f         = mask_train.astype(np.float64)
    obs_train_cms_filled = np.where(mask_train, Q_obs_train_cms, 0.0)

    valid_obs_train = Q_obs_train_cms[mask_train]
    if valid_obs_train.size == 0:
        print("⚠️  Skipped — no valid training observations after masking.")
        n_skip_no_data += 1
        continue

    denom_train_cms = np.sum((valid_obs_train - valid_obs_train.mean()) ** 2)

    # ── Multi-start calibration ───────────────────────────────────────────────
    best_fun = np.inf
    best_x   = None

    for _ in range(20):
        x0  = np.random.uniform(bounds_low, bounds_high)
        res = minimize(
            objective_hbv,
            x0,
            args=(P_train, ET_train, obs_train_cms_filled, mask_train_f, denom_train_cms, area),
            method="L-BFGS-B",
            bounds=param_bounds,
            options={"maxiter": 3000},
        )
        if res.fun < best_fun:
            best_fun = res.fun
            best_x   = res.x

    if best_x is None:
        print("⚠️  Skipped — calibration failed.")
        n_skip_calib_fail += 1
        continue

    # ── Full-period simulation ────────────────────────────────────────────────
    P_full   = np.asarray(P, dtype=np.float64)
    ET_full  = np.asarray(ET, dtype=np.float64)
    Qsim_mmd = np.empty(len(P_full), dtype=np.float64)
    hbv_numba(P_full, ET_full, best_x.astype(np.float64), Qsim_mmd)
    Qsim_cms = Qsim_mmd * area / CONV

    # ── Base Metrics ──────────────────────────────────────────────────────────
    m_train = compute_all_metrics(Q_obs_cms[:b1], Qsim_cms[:b1])
    m_val   = compute_all_metrics(Q_obs_cms[b1:], Qsim_cms[b1:])

    pte_train = compute_pte(Q_obs_cms[:b1], Qsim_cms[:b1], times[:b1])
    pte_val   = compute_pte(Q_obs_cms[b1:], Qsim_cms[b1:], times[b1:])

    # ── Seasonal Metrics (Train & Validation) ─────────────────────────────────
    months = pd.DatetimeIndex(times).month
    seasons = {"DJF": [12, 1, 2], "MAM": [3, 4, 5], "JJA": [6, 7, 8], "SON": [9, 10, 11]}
    seasonal_metrics = {}

    for season, m_list in seasons.items():
        mask_season = np.isin(months, m_list)

        mask_season_train = mask_season[:b1]
        mask_season_val   = mask_season[b1:]

        Q_obs_s_train, Q_sim_s_train = Q_obs_cms[:b1][mask_season_train], Qsim_cms[:b1][mask_season_train]
        Q_obs_s_val,   Q_sim_s_val   = Q_obs_cms[b1:][mask_season_val],   Qsim_cms[b1:][mask_season_val]

        valid_train = ~np.isnan(Q_obs_s_train)
        valid_val   = ~np.isnan(Q_obs_s_val)

        m_s_train = compute_all_metrics(Q_obs_s_train[valid_train], Q_sim_s_train[valid_train]) if np.sum(valid_train) > 10 else {}
        m_s_val = compute_all_metrics(Q_obs_s_val[valid_val], Q_sim_s_val[valid_val]) if np.sum(valid_val) > 10 else {}

        for metric in ["NSE", "KGE", "RMSE", "PBIAS"]:
            seasonal_metrics[f"{metric}_{season}_train"] = m_s_train.get(metric, np.nan)
            seasonal_metrics[f"{metric}_{season}_val"]   = m_s_val.get(metric, np.nan)

    # ── Console Output ────────────────────────────────────────────────────────
    print(f"✅ NSE   — train: {m_train['NSE']:.3f}   |  val: {m_val['NSE']:.3f}")
    print(f"   KGE   — train: {m_train['KGE']:.3f}   |  val: {m_val['KGE']:.3f}")
    print(f"   RMSE  — train: {m_train['RMSE']:.4f} m³/s  |  val: {m_val['RMSE']:.4f} m³/s")
    print(f"   PBIAS — train: {m_train['PBIAS']:.2f}%  |  val: {m_val['PBIAS']:.2f}%")
    print(f"   FHV   — train: {m_train['FHV']:.2f}%  |  val: {m_val['FHV']:.2f}%")
    print(f"   FLV   — train: {m_train['FLV']:.2f}%  |  val: {m_val['FLV']:.2f}%")
    print(f"   PTE   — train: {pte_train:.1f} days  |  val: {pte_val:.1f} days")
    print(f"   Runoff ratio : {RR:.3f}")
    print(f"   Best params  : {dict(zip(param_names, np.round(best_x, 3)))}")
    print("   ── Seasonal Validation (Train | Val) ──")
    for s in seasons.keys():
        print(f"      {s}  NSE: {seasonal_metrics[f'NSE_{s}_train']:.3f} | {seasonal_metrics[f'NSE_{s}_val']:.3f}    "
              f"PBIAS: {seasonal_metrics[f'PBIAS_{s}_train']:.2f}% | {seasonal_metrics[f'PBIAS_{s}_val']:.2f}%")

    # ── Store Results ─────────────────────────────────────────────────────────
    results[station_id] = {
        "params":        best_x.tolist(),
        "param_names":   param_names,
        "RR":            RR,
        "NSE_train":     m_train["NSE"],   "NSE_val":   m_val["NSE"],
        "NNSE_train":    m_train["NNSE"],  "NNSE_val":  m_val["NNSE"],
        "RMSE_train":    m_train["RMSE"],  "RMSE_val":  m_val["RMSE"],
        "PBIAS_train":   m_train["PBIAS"], "PBIAS_val": m_val["PBIAS"],
        "FHV_train":     m_train["FHV"],   "FHV_val":   m_val["FHV"],
        "FLV_train":     m_train["FLV"],   "FLV_val":   m_val["FLV"],
        "KGE_train":     m_train["KGE"],   "KGE_val":   m_val["KGE"],
        "PTE_train":     pte_train,        "PTE_val":   pte_val,
        **seasonal_metrics,
        "Qsim_mmd":      Qsim_mmd,
        "Qsim_cms":      Qsim_cms,
        "Q_obs_cms":     Q_obs_cms,
        "missing_ratio": missing_ratio,
        "missing_count": missing_count,
    }

# ============================================
# Summary report
# ============================================
n_total   = len(all_stations)
n_success = len(results)

print(f"\n{'='*52}")
print(f"  Calibration summary")
print(f"{'='*52}")
print(f"  Total stations              : {n_total:>5d}")
print(f"  Calibrated successfully     : {n_success:>5d}")
print(f"{'─'*52}")
print(f"  Skipped — no/invalid area   : {n_skip_no_area:>5d}")
print(f"  Skipped — too many NaN      : {n_skip_missing:>5d}")
print(f"  Skipped — no valid obs      : {n_skip_no_data:>5d}")
print(f"  Skipped — calibration fail  : {n_skip_calib_fail:>5d}")
print(f"{'─'*52}")
print(f"  Total skipped               : {n_total - n_success:>5d}")
print(f"{'='*52}")

Streaming output truncated to the last 5000 lines.
=== Station K223402001  (284/654) ===
✅ NSE   — train: 0.577   |  val: 0.693
   KGE   — train: 0.677   |  val: 0.669
   RMSE  — train: 0.3955 m³/s  |  val: 0.3325 m³/s
   PBIAS — train: -10.02%  |  val: -6.99%
   FHV   — train: -17.13%  |  val: -27.69%
   FLV   — train: -82.36%  |  val: 59.30%
   PTE   — train: 53.8 days  |  val: 9.8 days
   Runoff ratio : 0.526
   Best params  : {'FC': np.float64(203.028), 'BETA': np.float64(2.861), 'LP': np.float64(1.0), 'K0': np.float64(0.05), 'K1': np.float64(0.073), 'K2': np.float64(0.034), 'UZL': np.float64(100.0), 'PERC': np.float64(3.403)}
   ── Seasonal Validation (Train | Val) ──
      DJF  NSE: 0.207 | 0.527    PBIAS: -17.61% | -21.40%
      MAM  NSE: 0.455 | 0.599    PBIAS: -24.25% | -12.09%
      JJA  NSE: 0.655 | 0.700    PBIAS: 17.24% | 20.63%
      SON  NSE: 0.639 | 0.667    PBIAS: 15.53% | 13.70%

=== Station K225401001  (285/654) ===
✅ NSE   — train: 0.669   |  val: 0.768
   KGE   — t

## 5. SAVE SUMMARY

In [ ]:
import os
import pandas as pd
import numpy as np
from google.colab import drive

# ============================================
# 1. EXTRACT DATA TO DATAFRAME
# ============================================
rows = []
seasons = ["DJF", "MAM", "JJA", "SON"]

for station_id, res in results.items():
    # Dynamically unpack parameters using param_names
    param_dict = dict(zip(res["param_names"], res["params"]))

    row = {
        "station_id": station_id,
        **param_dict,
        "RR": res.get("RR", np.nan),
        "NSE_train": res.get("NSE_train", np.nan),
        "NSE_val": res.get("NSE_val", np.nan),
        "NNSE_train": res.get("NNSE_train", np.nan),
        "NNSE_val": res.get("NNSE_val", np.nan),
        "RMSE_train": res.get("RMSE_train", np.nan),
        "RMSE_val": res.get("RMSE_val", np.nan),
        "PBIAS_train": res.get("PBIAS_train", np.nan),
        "PBIAS_val": res.get("PBIAS_val", np.nan),
        "FHV_train": res.get("FHV_train", np.nan),
        "FHV_val": res.get("FHV_val", np.nan),
        "FLV_train": res.get("FLV_train", np.nan),
        "FLV_val": res.get("FLV_val", np.nan),
        "KGE_train": res.get("KGE_train", np.nan),
        "KGE_val": res.get("KGE_val", np.nan),
        "PTE_train": res.get("PTE_train", np.nan),
        "PTE_val": res.get("PTE_val", np.nan),
        "missing_ratio": res.get("missing_ratio", np.nan),
        "missing_count": res.get("missing_count", np.nan),
    }

    # Dynamically add seasonal metrics
    for season in seasons:
        row[f"NSE_{season}_train"]   = res.get(f"NSE_{season}_train", np.nan)
        row[f"NSE_{season}_val"]     = res.get(f"NSE_{season}_val", np.nan)
        row[f"KGE_{season}_train"]   = res.get(f"KGE_{season}_train", np.nan)
        row[f"KGE_{season}_val"]     = res.get(f"KGE_{season}_val", np.nan)
        row[f"RMSE_{season}_train"]  = res.get(f"RMSE_{season}_train", np.nan)
        row[f"RMSE_{season}_val"]    = res.get(f"RMSE_{season}_val", np.nan)
        row[f"PBIAS_{season}_train"] = res.get(f"PBIAS_{season}_train", np.nan)
        row[f"PBIAS_{season}_val"]   = res.get(f"PBIAS_{season}_val", np.nan)

    rows.append(row)

df = pd.DataFrame(rows)

# ============================================
# 2. SAVE RESULTS
# ============================================
# Local save
df.to_csv("HBV_Simulation_Data_CAMELS_FR.csv", index=False)
print("✅ Saved locally to HBV_Simulation_Data_CAMELS_FR.csv")

# Save to Google Drive (this is the file the loader script reads back)
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount("/content/drive")

df.to_csv("/content/drive/MyDrive/Colab Notebooks/Data/HBV_Simulation_Data_CAMELS_FR.csv", index=False)
print("✅ Saved to Google Drive")

# ============================================
# 3. DEFINE METRICS TO REPORT
# ============================================
global_metrics = [
    ("NSE_train",   "NSE Training",     ""),
    ("NSE_val",     "NSE Validation",   ""),
    ("NNSE_train",  "NNSE Training",    ""),
    ("NNSE_val",    "NNSE Validation",  ""),
    ("KGE_train",   "KGE Training",     ""),
    ("KGE_val",     "KGE Validation",   ""),
    ("RMSE_train",  "RMSE Training",    " m³/s"),
    ("RMSE_val",    "RMSE Validation",  " m³/s"),
    ("PBIAS_train", "PBIAS Training",   "%"),
    ("PBIAS_val",   "PBIAS Validation", "%"),
    ("FHV_train",   "FHV Training",     "%"),
    ("FHV_val",     "FHV Validation",   "%"),
    ("FLV_train",   "FLV Training",     "%"),
    ("FLV_val",     "FLV Validation",   "%"),
    ("PTE_train",   "PTE Training",     " days"),
    ("PTE_val",     "PTE Validation",   " days"),
]

season_vars = [("NSE", ""), ("KGE", ""), ("RMSE", " m³/s"), ("PBIAS", "%")]
seasonal_metrics = []
for var, unit in season_vars:
    for s in seasons:
        for split, label in [("train", "Training"), ("val", "Validation")]:
            seasonal_metrics.append((f"{var}_{s}_{split}", f"{var} {s} {label}", unit))

# ============================================
# 4. PRINT STATISTICS
# ============================================
def print_stats(metrics_list, df, title):
    print(f"\n{'='*60}")
    print(f" {title}")
    print(f"{'='*60}")
    for key, label, unit in metrics_list:
        if key not in df.columns:
            print(f"  ⚠️  '{key}' not found in CSV — skipping.")
            continue
        values = pd.to_numeric(df[key], errors="coerce").dropna().to_numpy()
        n_valid   = len(values)
        n_dropped = len(df) - n_valid
        if n_valid == 0:
            print(f"  ⚠️  {label}: all NaN — no valid stations.")
            continue
        print(
            f"  {label:<28} | "
            f"Mean: {values.mean():8.3f}{unit}  "
            f"Median: {np.median(values):8.3f}{unit}  "
            f"Min: {values.min():8.3f}{unit}  "
            f"Max: {values.max():8.3f}{unit}  "
            f"P05: {np.percentile(values,  5):8.3f}{unit}  "
            f"P95: {np.percentile(values, 95):8.3f}{unit}  "
            f"(n={n_valid})"
        )
        if n_dropped:
            print(f"    ⚠️  {n_dropped} station(s) excluded (NaN).")

print(f"\n✅ Saved {len(df)} stations to CSV.")
print_stats(global_metrics,   df, "GLOBAL SUMMARY STATISTICS")
print_stats(seasonal_metrics, df, "SEASONAL SUMMARY STATISTICS")

✅ Saved locally to HBV_Simulation_Data_CAMELS_FR.csv
✅ Saved to Google Drive

✅ Saved 549 stations to CSV.

 GLOBAL SUMMARY STATISTICS
  NSE Training                 | Mean:    0.669  Median:    0.730  Min:  -13.594  Max:    0.908  P05:    0.447  P95:    0.849  (n=549)
  NSE Validation               | Mean:    0.625  Median:    0.741  Min:  -37.356  Max:    0.942  P05:    0.340  P95:    0.886  (n=549)
  NNSE Training                | Mean:    0.774  Median:    0.788  Min:    0.064  Max:    0.916  P05:    0.644  P95:    0.869  (n=549)
  NNSE Validation              | Mean:    0.779  Median:    0.794  Min:    0.025  Max:    0.945  P05:    0.602  P95:    0.898  (n=549)
  KGE Training                 | Mean:    0.753  Median:    0.793  Min:   -3.636  Max:    0.940  P05:    0.528  P95:    0.902  (n=549)
  KGE Validation               | Mean:    0.672  Median:    0.733  Min:   -7.375  Max:    0.951  P05:    0.362  P95:    0.899  (n=549)
  RMSE Training                | Mean:    4.529 m³/s  M